## Qutip Tutorial

#### Imports

In [2]:
import qutip
from qutip import operators
from typing import List,Tuple
import numpy as np

#### First: Define the transverse ising model 

In qutip the spin basis is simply a Fock representation with 2 D.O.F (Hardcore bosons). If we want to write the hamiltonian we should consider the tensor product operator for each term (it is not like quspin).

#### Define the general operator method

We implement a function in order to compute the general many-body operator

$O^{a_1,a_2,...,a_n}_{i_1, i_2, ..., i_n}=G^{a_1, a_2, .. a_n}_{i_1,i_2,...,i_n} s^{a_1}_{i_1} \otimes s^{a_2}_{i_2} \otimes ... \otimes s^{a_n}_{i_n}$

In [8]:


def manybodyoperator(directions:List[List],size:int)->qutip.Qobj:

    pauli={'idx':qutip.identity(2),'x':qutip.sigmax(),'y':qutip.sigmay(),'z':qutip.sigmaz(),'+':qutip.sigmap,'-':qutip.sigmam}
    # for each coupling term in the list direction
    for r,direction in enumerate(directions):
        coupling=direction[0]
        
        # starting point -> identity operator
        idx_mb:List[str]=['idx' for i in range(size)]
        
        # create the op representation
        for dir,i in direction:
            #print('dir=',dir,'i=',i)
            idx_mb[i]=dir
        # convert into qutip.Qobj
        for i in range(size):
            if i==0:
                op=pauli[idx_mb[i]]        
            else:
                op=qutip.tensor(op,pauli[idx_mb[i]])
        #sum each direction
        if r==0:
            manybodyop=op*coupling
        else:
            manybodyop=manybodyop+op*coupling
                
    return manybodyop


#### Test the Method

We study the operator $O_{i ,i+1}=\sum^{N-1}_i x_i \otimes x_{i+1}$

In [9]:
size:int=3
coupling:List=[1]*size
directions:List[List]=[[coupling[i],('x',i),('x',i+1)] for i in range(size-1)]
op:qutip.Qobj=manybodyoperator(directions=directions,size=size)

print(op)

TypeError: cannot unpack non-iterable int object

#### Implement a SteadyState class

In [10]:

class SteadyStateClass():
    
    def __init__(self,size:int,unitary_list:List[List],dissipative_list:List[List]) -> None:
        
        #parameters
        self.unitary_list:List[List]=unitary_list
        self.dissipative_list:List[List]=dissipative_list
        self.size=size
        
        #attributes
        self.steady_state:qutip.Qobj=None
        self.limbladian:qutip.Qobj=None
    
    
    #operation that convert the abstract string to the qutip.Qobj    
    def _manybodyoperator(self,directions:List[List],size:int)->qutip.Qobj:
        #pauli dictionary
        pauli={'idx':qutip.identity(2),'x':qutip.sigmax(),'y':qutip.sigmay(),'z':qutip.sigmaz(),'+':qutip.sigmap,'-':qutip.sigmam}

        # for each coupling term in the list direction
        for r,direction in enumerate(directions):
            coupling=direction[0] #coupling term in the direction list
            # starting point -> identity operator
            idx_mb:List[str]=['idx' for i in range(size)]
            
            # create the op representation
            for dir,i in direction[1:]:
                #print('dir=',dir,'i=',i)
                idx_mb[i]=dir
            # convert into qutip.Qobj
            for i in range(size):
                if i==0:
                    op=pauli[idx_mb[i]]        
                else:
                    op=qutip.tensor(op,pauli[idx_mb[i]])
            #sum each direction
            if r==0:
                manybodyop=op*coupling
            else:
                manybodyop=manybodyop+op*coupling
                    
        return manybodyop
    
    def _get_the_limbladian(self)->None:
        #define the hamiltonian
        for i,u in enumerate(self.unitary_list):
            if i==0:
                hamiltonian=self._manybodyoperator(directions=u,size=self.size)
            else:
                hamiltonian=hamiltonian+self._manybodyoperator(directions=u,size=self.size)
        dissipative=[]
        for d in self.dissipative_list:
            dissipative.append(self._manybodyoperator(directions=d,size=self.size))
        self.limbladian=qutip.liouvillian(H=hamiltonian,c_ops=dissipative)
        
    def get_steady_state(self)->qutip.Qobj:
        self._get_the_limbladian()
        self.steady_state=qutip.steadystate(qutip.to_super(self.limbladian))

    
    def print_liouvillian(self)->None:
        print('Unitary part=\n',self.unitary_list,'\n')
        print('Dissipative part=\n',self.dissipative_list,'\n')
        
    def steady_state_expect(self,directions:List[List])->float:
        # define the operator in 
        op=self._manybodyoperator(directions=directions,size=self.size)
        return qutip.expect(op,self.steady_state)        
    
    

#### Check if the SteadyState Class works

In [12]:
size:int=3
j:float=1
h:float=1
g:float=0.1
xx:List[List]=[[j,('x',i),('x',i+1)] for i in range(size-1)]
z:List[List]=[[h,('z',i)] for i in range(size)]
x:List[List]=[[g,('x',i)] for i in range(size)]

unitary=[xx,z]
dissipative=[]

std=SteadyStateClass(size=size,unitary_list=unitary,dissipative_list=dissipative)

std.print_liouvillian()

std.get_steady_state()

print(std.steady_state)

print(std.steady_state_expect(xx+z))





Unitary part=
 [[[1, ('x', 0), ('x', 1)], [1, ('x', 1), ('x', 2)]], [[1, ('z', 0)], [1, ('z', 1)], [1, ('z', 2)]]] 

Dissipative part=
 [] 

Quantum object: dims = [[2, 2, 2], [2, 2, 2]], shape = (8, 8), type = oper, isherm = True
Qobj data =
[[ 9.79125837e-07+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j -9.09081556e-02+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j -4.54542537e-02-1.35525272e-20j
  -9.09081556e-02+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 [ 0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 [ 0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j
   0.00000000e+00+0.00000000e

#### Another method for the directions

In [46]:
op={}
index=(0,1,2)
coupling=1
direction=['x','x','x']
op[index]={"coupling":coupling, "direction":direction}

class AbstractOperator():
    def __init__(self,index:List,direction:List[List],coupling:List,size:int) -> None:
        
        self.index=index
        self.direction=direction
        self.coupling=coupling
        self.size=size
        # if indices> size the operator is ill defined
        print(max([max(idx) for idx in self.index]))
        assert max([max(idx) for idx in self.index])<=(self.size-1),f"operator defined in a larger size system: idx > l={self.size}" 
        
        self.op:dict={}
        self.__get_operator()
    
    def __get_operator(self):
        for i,idx in enumerate(self.index):
            self.op[idx]={"coupling":self.coupling[i],"direction":self.direction[i]}
    
    def printout(self):
        print(self.op)
        
    def abstract2qutip(self):
        #operation that convert the abstract string to the qutip.Qobj    
        #pauli dictionary
        pauli={'idx':qutip.identity(2),'x':qutip.sigmax(),'y':qutip.sigmay(),'z':qutip.sigmaz(),'+':qutip.sigmap,'-':qutip.sigmam}

        qutip_op:qutip.Qobj=0
        # create the op representation
        for index in self.op.keys():
            # starting point -> identity operator
            idx_mb:List[str]=['idx' for i in range(self.size)]
            for j,idx in enumerate(index):
                idx_mb[idx]=self.op[index]['direction'][j]
        # convert into qutip.Qobj
            for i in range(self.size):
                if i==0:
                    op=pauli[idx_mb[i]]        
                else:
                    op=qutip.tensor(op,pauli[idx_mb[i]])
            #sum each direction
            qutip_op=qutip_op+op*self.op[index]['coupling']
                
        return qutip_op
    

class SpinHamiltonian(AbstractOperator):
    def __init__(self,interaction:AbstractOperator,ext_field:AbstractOperator) -> None:
        super.__init__()
        self.interaction=interaction
        self.ext_field=ext_field
        self.op=dict(interaction).update(ext_field)
    
    def printout(self):
        print('interaction:\n')
        self.interaction.printout()
        print('external field:\n')
        self.ext_field.printout()
        
        
            

#### Example of AbstractOperator

In [47]:
l=2
index_xx=[(0,1)]
direction=[['x','x']]
coupling=[0.5]

xx=AbstractOperator(index=index_xx,direction=direction,coupling=coupling,size=l)

xx.printout()

xx_qutip=xx.abstract2qutip()

print(xx_qutip)


1
{(0, 1): {'coupling': 0.5, 'direction': ['x', 'x']}}
Quantum object: dims = [[2, 2], [2, 2]], shape = (4, 4), type = oper, isherm = True
Qobj data =
[[0.  0.  0.  0.5]
 [0.  0.  0.5 0. ]
 [0.  0.5 0.  0. ]
 [0.5 0.  0.  0. ]]


#### Create the Spin Hamiltonian for Abstract operators

In [48]:
l=2

index_xx=[[(0,1)]]
direction_xx=[['x','x']]
coupling=[0.5]
interaction=AbstractOperator(index_xx,direction_xx,coupling,size=l)

index_z=[(0),(1)]
direction_z=[['z']*l]
coupling=[[1]*l]
ext_field=AbstractOperator(index_z,direction_z,coupling=coupling,size=l)

#create the Spin Hamiltonian
hamiltonian=SpinHamiltonian(interaction=interaction,ext_field=ext_field,size=l)


hamiltonian.printout()


(0, 1)


TypeError: '<=' not supported between instances of 'tuple' and 'int'